# 2 — Map merged D2 and D4 queries onto the D10 SCANVI reference

Per timepoint, concatenate the Dz + Lapa slim files and map the merged query against the frozen reference. Produces a single integrated latent space per timepoint that contains both treatments.

Outputs:
- `data/data-objects/cellassign/d2_scanvi_predictions.h5ad`  (D2_DZ + D2_Lapa cells)
- `data/data-objects/cellassign/d4_scanvi_predictions.h5ad`  (D4_DZ + D4_Lapa cells)
- Probability CSVs alongside in `data/analysis-files/cell_assign_dfs/`

Set `ACCELERATOR='gpu'` on HPC.

In [ ]:
import sys, gc
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import INTEGRATION_DIR, CELLASSIGN_DIR, ANALYSIS_FILES_DIR
from src.transfer_learning import map_query

import scanpy as sc
import anndata as ad

ACCELERATOR = 'auto'  # 'gpu' on HPC; 'auto' picks GPU if present
AMBIGUOUS_THRESHOLD = 0.5
MAX_EPOCHS_QUERY = 200

TIMEPOINT_GROUPS = {
    'D2': ['D2_DZ', 'D2_Lapa'],
    'D4': ['D4_DZ', 'D4_Lapa'],
}

SLIM_HVG = INTEGRATION_DIR / 'slim_hvg'
MERGED_DIR = INTEGRATION_DIR / 'merged_query'
MODEL_DIR = INTEGRATION_DIR / 'scarches_model'
CSV_DIR = ANALYSIS_FILES_DIR / 'cell_assign_dfs'
MERGED_DIR.mkdir(parents=True, exist_ok=True)
CELLASSIGN_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Build the merged slim file per timepoint (Dz + Lapa concat).
# At HVG resolution this is small (~30k cells x 3000 genes sparse).
for tp, members in TIMEPOINT_GROUPS.items():
    parts = []
    for k in members:
        a = sc.read_h5ad(SLIM_HVG / f'{k}.h5ad')
        parts.append(a)
    merged = ad.concat(parts, axis=0, join='inner', label=None, merge='same')
    merged.obs_names_make_unique()
    out_path = MERGED_DIR / f'{tp}.h5ad'
    merged.write_h5ad(out_path, compression='gzip')
    print(f'{tp}: {merged.n_obs} cells x {merged.n_vars} genes -> {out_path}')
    print('  by dataset:', merged.obs["dataset"].value_counts().to_dict())
    print('  by treatment:', merged.obs["Treatment"].astype(str).value_counts().to_dict())
    del parts, merged, a
    gc.collect()

In [ ]:
# Map each merged timepoint query against the frozen SCANVI reference
results = {}
for tp in TIMEPOINT_GROUPS:
    print(f'\n=== mapping {tp} (Dz + Lapa merged) ===')
    out = map_query(
        query_slim_path=MERGED_DIR / f'{tp}.h5ad',
        ref_model_dir=MODEL_DIR,
        predictions_h5ad_path=CELLASSIGN_DIR / f'{tp.lower()}_scanvi_predictions.h5ad',
        predictions_csv_path=CSV_DIR / f'{tp.lower()}_scanvi_probabilities.csv',
        layer='counts',
        max_epochs_query=MAX_EPOCHS_QUERY,
        ambiguous_threshold=AMBIGUOUS_THRESHOLD,
        seed=0,
        accelerator=ACCELERATOR,
    )
    results[tp] = out
    gc.collect()
results

In [ ]:
# Per-timepoint summary: cells, mean confidence, % ambiguous, label distribution by treatment
import pandas as pd
for tp in TIMEPOINT_GROUPS:
    a = sc.read_h5ad(results[tp]['h5ad'])
    n = a.n_obs
    n_amb = a.obs['scanvi_prediction'].astype(str).str.startswith('Ambiguous_').sum()
    print(f'\n{tp}: {n} cells, mean conf={a.obs["scanvi_confidence"].mean():.3f}, %ambiguous={n_amb/n:.2%}')
    by_tx = pd.crosstab(a.obs['Treatment'].astype(str), a.obs['scanvi_prediction_raw'].astype(str), normalize='index').round(3)
    print(by_tx)
    del a; gc.collect()

In [ ]:
# Confidence histograms per timepoint
import matplotlib.pyplot as plt
from src.config import FIGURES_DIR
fig_dir = FIGURES_DIR / 'd2-d4-integrated' / 'sanity'
fig_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(TIMEPOINT_GROUPS), figsize=(4*len(TIMEPOINT_GROUPS), 3), sharey=True)
for ax, tp in zip(axes, TIMEPOINT_GROUPS):
    a = sc.read_h5ad(results[tp]['h5ad'])
    ax.hist(a.obs['scanvi_confidence'], bins=40)
    ax.axvline(AMBIGUOUS_THRESHOLD, color='red', linestyle='--', linewidth=1)
    ax.set_title(tp); ax.set_xlabel('confidence'); ax.set_xlim(0, 1)
    del a; gc.collect()
axes[0].set_ylabel('cells')
plt.tight_layout()
plt.savefig(fig_dir / 'confidence_histograms_timepoint.pdf')
plt.show()